# Random Forest after 10 full moves (20 plies)

双方各走10步。只训练随机森林，比较赛前6列与赛前＋开局46列。
本notebook读取已保存的模型开发和评估结果；Run All不会再次训练或评分测试集。
从头复现请使用根目录README中的命令和新的运行目录。


In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
assert (root / "src").exists(), "Open this notebook inside the Chess checkout"
run = root/'modelling_rf/run_20261025_20ply_01'
print(json.loads((root/'artifacts/plan_a20/processing_audit.json').read_text()))


{'status': 'completed', 'cutoff_ply': 20, 'source_clean_rows': 19111, 'decisions': {'finished_by_ply20': 1651, 'retained': 17050, 'original_boundary_gap': 410}, 'splits': [{'split': 'train', 'original_rows': 13229, 'retained_rows': 11936, 'first_start_utc': '2013-08-17 20:37:48.314000+00:00', 'last_start_utc': '2017-08-08 22:19:01.315000+00:00'}, {'split': 'validation', 'original_rows': 2747, 'retained_rows': 2549, 'first_start_utc': '2017-08-10 00:41:12.017000+00:00', 'last_start_utc': '2017-08-27 23:46:17.557000+00:00'}, {'split': 'test', 'original_rows': 2725, 'retained_rows': 2565, 'first_start_utc': '2017-08-29 00:04:08.817000+00:00', 'last_start_utc': '2017-09-04 02:45:43.790000+00:00'}], 'pregame_feature_count': 6, 'opening_feature_count': 40, 'combined_feature_count': 46, 'python': '3.12.14', 'packages': {'chess': '1.11.2', 'pandas': '2.2.3', 'numpy': '2.3.5', 'scikit-learn': '1.8.0'}, 'test_performance_evaluated': False}


## 训练内时间CV和验证

参数选择只用训练/验证信息；这是在旧测试已被查看之后的新探索，非全新盲测。

In [2]:
print(pd.read_csv(run/'tables/cv_summary.csv').to_string(index=False))
print(pd.read_csv(run/'tables/validation_selected_metrics.csv').to_string(index=False))
frozen = json.loads((run/'artifacts/frozen_config.json').read_text())
print('Frozen parameters:', frozen['parameters'])
print('Preferred before test:', frozen['preferred_feature_set_by_validation'])


 candidate feature_set  mean_cv_macro_f1  std_cv_macro_f1                                                                                                                        parameters
         0     pregame          0.416221         0.001797           {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 3, "max_features": "sqrt", "class_weight": null, "n_jobs": 4}
         0    combined          0.437440         0.005606           {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 3, "max_features": "sqrt", "class_weight": null, "n_jobs": 4}
         1     pregame          0.429091         0.001832     {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 3, "max_features": "sqrt", "class_weight": "balanced", "n_jobs": 4}
         1    combined          0.443494         0.002956     {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 3, "max_features": "sqrt", "class_weight": "balanced", "n_jobs": 4}
         2     pregame          0.416618         0.001033   

## 冻结测试结果

Accuracy及Macro-F1改善，但和棋Recall下降；不能单看总体准确率。

In [3]:
print(pd.read_csv(run/'tables/test_metrics.csv').to_string(index=False))
print(pd.read_csv(run/'tables/test_combined_per_class.csv',index_col=0).to_string())
print(pd.read_csv(run/'tables/test_combined_confusion.csv',index_col=0).to_string())


feature_set  test_rows  refit_rows  refit_seconds  accuracy  macro_f1  balanced_accuracy  draw_recall  log_loss
    pregame       2565       14485       0.427437  0.559454  0.417667           0.419208     0.083333  0.910133
   combined       2565       14485       0.507552  0.616374  0.449503           0.450219     0.053030  0.883630
              precision    recall  f1-score      support
black          0.596426  0.689922  0.639776  1161.000000
draw           0.127273  0.053030  0.074866   132.000000
white          0.662382  0.607704  0.633866  1272.000000
accuracy       0.616374  0.616374  0.616374     0.616374
macro avg      0.462027  0.450219  0.449503  2565.000000
weighted avg   0.604990  0.616374  0.607774  2565.000000
        black  draw  white
actual                    
black     801    18    342
draw       73     7     52
white     469    30    773


In [4]:
print(pd.read_csv(run/'tables/test_subgroup_metrics.csv').to_string(index=False))
print(pd.read_csv(run/'tables/validation_permutation_importance.csv').head(15).to_string(index=False))


feature_set                            group  rows  draw_support  accuracy  macro_f1  balanced_accuracy  draw_recall  log_loss
    pregame                 rating_gap_le100  1392            81  0.435345  0.339204           0.346259     0.123457  1.007238
    pregame                 rating_gap_gt100  1173            51  0.706735  0.494841           0.498673     0.019608  0.794897
    pregame     any_player_seen_in_final_fit  1896            96  0.545359  0.407292           0.408380     0.083333  0.918426
    pregame neither_player_seen_in_final_fit   669            36  0.599402  0.447042           0.451285     0.083333  0.886629
   combined                 rating_gap_le100  1392            81  0.528736  0.392302           0.394475     0.061728  0.958189
   combined                 rating_gap_gt100  1173            51  0.720375  0.513509           0.513936     0.039216  0.795151
   combined     any_player_seen_in_final_fit  1896            96  0.602848  0.430369           0.433336     0.0